# Laboratorio 00.1 — Preparación del entorno

En este laboratorio verificaremos el intérprete, las dependencias y la configuración local del curso sin llamar a ningún modelo.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Verificar que el kernel usa Python 3.11 o posterior.
- Identificar las dependencias instaladas y detectar las ausentes.
- Cargar la configuración YAML y validar sus contratos con Pydantic.
- Guardar un informe reproducible del entorno en formato CSV.

## 2. Conceptos

Un entorno reproducible registra tanto la versión de Python como las bibliotecas que ejecutan el experimento. El archivo `pyproject.toml` declara restricciones compatibles; `requirements.txt` ofrece una instalación directa. Pydantic valida la configuración cargada desde YAML antes de que llegue al código de aplicación. La presencia de una biblioteca no implica que haya credenciales, conexión a un proveedor o acceso a un modelo local.

## 3. Arquitectura

```mermaid
flowchart TD
    A[Kernel de Python] --> B[Auditoria de dependencias]
    B --> C[Configuracion YAML]
    C --> D[Validacion Pydantic]
    D --> E[Informe CSV]
    D -. sin llamadas .-> F[Proveedores LLM]
```

## 4. Preparación del entorno

Este bloque localiza la carpeta del curso tanto si Jupyter se inició aquí como si se inició en la raíz del repositorio. Carga `.env` sin mostrar sus valores y añade el paquete local al camino de importación.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

import pandas as pd

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))
from genai.environment import preparar_entorno

raiz_curso = preparar_entorno()
print(f'Raíz del curso: {raiz_curso}')

Raíz del curso: C:\Users\abelv\Documents\cursos\Elementos de IA\labs-elementos-ia\genai


**Auditoría de dependencias.** Este bloque consulta metadatos locales para conocer la versión de cada biblioteca requerida, sin importarla ni conectarse a servicios. Observa qué paquetes faltan antes de continuar y distingue una ausencia del entorno de un error de configuración.

In [5]:
paquetes_requeridos = [
    'pydantic', 'PyYAML', 'python-dotenv', 'pandas', 'numpy',
    'matplotlib', 'scikit-learn', 'jupyterlab', 'ipykernel',
    'langchain-core', 'langchain-ollama', 'langchain-openai',
    'langchain-anthropic', 'langgraph', 'langsmith',
]
filas_paquetes = []
for nombre_paquete in paquetes_requeridos:
    try:
        version_instalada = version(nombre_paquete)
        estado = 'instalado'
    except PackageNotFoundError:
        version_instalada = None
        estado = 'ausente'
    filas_paquetes.append({
        'paquete': nombre_paquete,
        'version': version_instalada,
        'estado': estado,
    })

auditoria = pd.DataFrame(filas_paquetes)
display(auditoria)

,paquete,version,estado
0,pydantic,2.11.7,instalado
1,PyYAML,6.0.3,instalado
2,python-dotenv,1.0.1,instalado
3,pandas,2.2.2,instalado
4,numpy,2.0.0,instalado
5,matplotlib,3.10.5,instalado
6,scikit-learn,1.7.2,instalado
7,jupyterlab,None,ausente
8,ipykernel,7.3.0,instalado
9,langchain-core,1.2.2,instalado


## 5. Definición del problema

Antes de comparar modelos necesitamos distinguir errores del sistema de errores del experimento. Este laboratorio define como problema una instalación incompleta o una configuración inválida. El conjunto de datos es el inventario local de versiones, obtenido desde los metadatos instalados; no se usa una API ni se envían datos fuera del equipo.

## 6. Línea base

La línea base es el propio intérprete. La versión mínima del curso es Python 3.11; el resultado se registra explícitamente para que una instalación incorrecta no quede confundida con un fallo de una biblioteca.

In [6]:
version_python = tuple(sys.version_info[:2])
linea_base = pd.DataFrame([{
    'python': sys.version.split()[0],
    'version_minima': '3.11',
    'cumple_requisito': version_python >= (3, 11),
}])
display(linea_base)

,python,version_minima,cumple_requisito
0,3.12.3,3.11,True


## 7. Implementación

Ahora validamos los archivos de configuración. El registro Pydantic limita los proveedores admitidos y verifica que cada uno tenga un identificador de modelo no vacío; observa el esquema JSON para inspeccionar ese contrato.

In [10]:
from genai.models import load_model_registry, load_runtime_settings

registro_modelos = load_model_registry(raiz_curso / 'config' / 'models.yaml')
ajustes = load_runtime_settings(raiz_curso / 'config' / 'settings.yaml')
print('Proveedores configurados:', sorted(registro_modelos.providers))
print('Temperatura predeterminada:', ajustes.temperature)
display(pd.DataFrame([
    {'proveedor': proveedor, 'modelo': configuracion.model}
    for proveedor, configuracion in registro_modelos.providers.items()
]))
display(registro_modelos.model_json_schema())

Proveedores configurados: ['anthropic', 'ollama', 'openai']
Temperatura predeterminada: 0.0


,proveedor,modelo
0,ollama,llama3.2
1,openai,gpt-4o-mini
2,anthropic,claude-3-5-sonnet-latest


{'$defs': {'ProviderConfig': {'additionalProperties': False,
   'description': 'Configuracion de un proveedor y su identificador de modelo.',
   'properties': {'model': {'minLength': 1,
     'title': 'Model',
     'type': 'string'},
    'base_url': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
     'default': None,
     'title': 'Base Url'}},
   'required': ['model'],
   'title': 'ProviderConfig',
   'type': 'object'}},
 'additionalProperties': False,
 'description': 'Registro de modelos que se carga desde `config/models.yaml`.',
 'properties': {'providers': {'additionalProperties': {'$ref': '#/$defs/ProviderConfig'},
   'propertyNames': {'enum': ['ollama', 'openai', 'anthropic']},
   'title': 'Providers',
   'type': 'object'}},
 'required': ['providers'],
 'title': 'ModelRegistry',
 'type': 'object'}

## 8. Experimentos

Este experimento compara dos temperaturas en la configuración, manteniendo iguales el proveedor y el modelo. No genera texto: aislar parámetros antes de gastar llamadas permite diseñar un experimento controlado.

In [11]:
from genai.models import RuntimeSettings

experimento_temperatura = pd.DataFrame([
    {'temperatura': valor, 'configuracion_valida': RuntimeSettings(temperature=valor).temperature}
    for valor in (0.0, 0.7)
])
display(experimento_temperatura)

,temperatura,configuracion_valida
0,0.0,0.0
1,0.7,0.7


## 9. Evaluación

Cuantificamos cobertura de configuración y disponibilidad de paquetes. La cobertura solo informa si los tres proveedores están descritos; no afirma que sus servicios estén accesibles. El CSV deja un registro local que se puede comparar entre entornos.

In [12]:
proveedores_esperados = {'ollama', 'openai', 'anthropic'}
proveedores_configurados = set(registro_modelos.providers)
metricas_entorno = pd.DataFrame([{
    'python_valido': version_python >= (3, 11),
    'paquetes_instalados': int((auditoria['estado'] == 'instalado').sum()),
    'paquetes_requeridos': len(auditoria),
    'cobertura_proveedores': len(proveedores_configurados & proveedores_esperados) / len(proveedores_esperados),
}])
carpeta_salida = raiz_curso / 'outputs'
carpeta_salida.mkdir(parents=True, exist_ok=True)
auditoria.to_csv(carpeta_salida / '00_01_dependency_check.csv', index=False)
metricas_entorno.to_csv(carpeta_salida / '00_01_environment_metrics.csv', index=False)
display(metricas_entorno)

,python_valido,paquetes_instalados,paquetes_requeridos,cobertura_proveedores
0,True,12,15,1.0


## 10. Discusión

Una biblioteca instalada no garantiza que la versión sea compatible con cualquier servicio, y una configuración válida no equivale a una conexión probada. Si el inventario muestra ausencias, instala las dependencias del curso y vuelve a ejecutar. Las credenciales y el servidor local se verificarán solo cuando un experimento los necesite.

## 11. Ejercicios

1. Añade al informe la plataforma del sistema operativo y explica por qué puede afectar la instalación.
2. Cambia un identificador de modelo en `models.yaml` y comprueba que el informe refleja el cambio.
3. Provoca un error de validación con una temperatura fuera del intervalo permitido y describe el error de Pydantic.

## 12. Desafío

Extiende el informe para comparar dos entornos Python distintos y detectar diferencias de versiones que puedan afectar a un mismo experimento, sin incluir credenciales ni datos personales.

## 13. Ideas clave

- Registrar versiones reduce la ambigüedad al depurar.
- Pydantic convierte archivos de configuración en contratos comprobables.
- La configuración del proveedor no prueba conectividad ni disponibilidad.
- Este laboratorio no realiza llamadas a modelos.